In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, lower, current_timestamp, concat, lpad, lit, when, trim
from pyspark.sql.types import StringType, BooleanType
import re
import time

#The second line (with 4g) specifies how much RAM to use. change according to machine
spark = SparkSession.builder \
    .appName("IngestionFramework") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()



# equivalent to a JSON config file - the only non-modular part.
# Has entries that are specific to each dataset, like:

# primary_keys: if its a list then its the combination of these columns

# timestamp_cols: dict with columns cast to TimestampType (after snake_case rename).

# assembled_timestamp : dict with keys year_col, month_col, day_col,
#                         and optionally hour_col, minute_col, second_col,
#                         plus output_col for the new timestamp column name.
#                         Used when date parts live in separate integer columns.
#
# date_time_pairs     : list of {date_col, time_col, output_col, format}
#                         Used when date and time live in separate string columns.
#
# bool_cols           : list of column names to cast to BooleanType.
#                         Understands Y/N, yes/no, true/false, 1/0.
DATASET_CONFIGS = {
    "weather": {
        "format": "csv",
        "path": "datasets/weather.csv",
        "primary_keys": ["year", "month", "day", "hour"],
        "timestamp_cols": {},
        # year/month/day/hour integer columns into single timestamp column
        "assembled_timestamp": {
            "year_col":   "year",
            "month_col":  "month",
            "day_col":    "day",
            "hour_col":   "hour",
            "output_col": "timestamp"
        },
        "date_time_pairs": [],
        "bool_cols": [],
        "options": {"header": "true", "inferSchema": "true"}
    },
    "taxi_trips_01": {
        "format": "parquet",
        "path": "datasets/yellow_tripdata_2024-01.parquet",
        "primary_keys": [],
        "timestamp_cols": {
            "tpep_pickup_datetime":  "yyyy-MM-dd HH:mm:ss",
            "tpep_dropoff_datetime": "yyyy-MM-dd HH:mm:ss"
        },
        "assembled_timestamp": None,
        "date_time_pairs": [],
        # store_and_fwd_flag is "Y"/"N" in trip data
        "bool_cols": ["store_and_fwd_flag"],
        "options": {"header": "true", "inferSchema": "true"}
    },
    "taxi_trips_02": {
        "format": "parquet",
        "path": "datasets/yellow_tripdata_2024-02.parquet",
        "primary_keys": [],
        "timestamp_cols": {
            "tpep_pickup_datetime":  "yyyy-MM-dd HH:mm:ss",
            "tpep_dropoff_datetime": "yyyy-MM-dd HH:mm:ss"
        },
        "assembled_timestamp": None,
        "date_time_pairs": [],
        "bool_cols": ["store_and_fwd_flag"],
        "options": {"header": "true", "inferSchema": "true"}
    },
    "taxi_trips_03": {
        "format": "parquet",
        "path": "datasets/yellow_tripdata_2024-03.parquet",
        "primary_keys": [],
        "timestamp_cols": {
            "tpep_pickup_datetime":  "yyyy-MM-dd HH:mm:ss",
            "tpep_dropoff_datetime": "yyyy-MM-dd HH:mm:ss"
        },
        "assembled_timestamp": None,
        "date_time_pairs": [],
        "bool_cols": ["store_and_fwd_flag"],
        "options": {"header": "true", "inferSchema": "true"}
    },
    "taxi_zone_lookup": {
        "format": "csv",
        "path": "datasets/taxi_zone_lookup.csv",
        "primary_keys": ["location_id"],
        "timestamp_cols": {},
        "assembled_timestamp": None,
        "date_time_pairs": [],
        "bool_cols": [],
        "options": {"header": "true", "inferSchema": "true"}
    },
    "air_quality": {
        "format": "csv",
        "path": "datasets/air_quality.csv",
        "primary_keys": [
            "state_code", "county_code", "site_num",
            "parameter_code", "poc", "date_local", "time_local"
        ],
        "timestamp_cols": {},
        "assembled_timestamp": None,
        # Combine the separate date + time string columns into timestamps
        "date_time_pairs": [
            {
                "date_col":   "date_local",
                "time_col":   "time_local",
                "output_col": "datetime_local",
                "format":     "yyyy-MM-dd HH:mm"
            },
            
        ],
        "bool_cols": [],
        "options": {"header": "true", "inferSchema": "true"}
        }
}

#-----------------------------Helpers-------------------

# Helper - column name to snake case

def to_snake_case(name):
    """CamelCase / mixed case -> snake_case, handles acronym runs like 'PULocationID' and spaces/dashes."""
    s = name.strip()
    s = re.sub(r'[\s\-]+', '_', s)                     # Replace spaces/dashes with single underscore first
    s1 = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', s)   # Handle runs of caps (e.g. PULocation -> PU_Location)
    s2 = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1)    # Handle lower to upper transitions
    s3 = re.sub(r'_+', '_', s2)                        # Collapse any consecutive underscores
    return s3.lower()


def standardize_columns(df):
    for c in df.columns:
        df = df.withColumnRenamed(c, to_snake_case(c))
    return df



# Timestamp normalisation to format yyyy-mm-dd HH:mm:ss
#3 types of normalization

# Cast timestamp to TimeStampType
def normalize_timestamps(df, timestamp_cols):
    
    for col_name, fmt in timestamp_cols.items():
        snake = to_snake_case(col_name)
        # If the column is already a TimestampType (e.g. from Parquet), keep it;
        # otherwise parse using the supplied format string.
        col_dtype = dict(df.dtypes).get(snake)
        if col_dtype == "timestamp":
            pass  
        else:
            df = df.withColumn(snake, to_timestamp(col(snake), fmt))
    return df


# Build a single TimestampType column from separate integer or string
# year / month / day / [hour / minute / second] columns.
# Used for weather dataset
def assemble_timestamp_from_parts(df, config):
    if not config:
        return df

    year   = config["year_col"]
    month  = config["month_col"]
    day    = config["day_col"]
    hour   = config.get("hour_col")
    minute = config.get("minute_col")
    second = config.get("second_col")
    out    = config["output_col"]

    # Build a string like "2024-01-01 00:00:00" then parse it
    time_part = concat(
        lpad(col(hour).cast("string"),   2, "0") if hour   else lit("00"), lit(":"),
        lpad(col(minute).cast("string"), 2, "0") if minute else lit("00"), lit(":"),
        lpad(col(second).cast("string"), 2, "0") if second else lit("00"),
    )
    date_part = concat(
        col(year).cast("string"),  lit("-"),
        lpad(col(month).cast("string"), 2, "0"), lit("-"),
        lpad(col(day).cast("string"),   2, "0"),
    )
    datetime_str = concat(date_part, lit(" "), time_part)
    df = df.withColumn(out, to_timestamp(datetime_str, "yyyy-MM-dd HH:mm:ss"))
    return df

# Combine separate date-string and time-string columns into a single column
# Used for air_quality dataset
def normalize_date_time_pairs(df, pairs):
    for pair in pairs:
        date_col   = pair["date_col"]
        time_col   = pair["time_col"]
        output_col = pair["output_col"]
        fmt        = pair["format"]
        datetime_str = concat(col(date_col), lit(" "), col(time_col))
        df = df.withColumn(output_col, to_timestamp(datetime_str, fmt))
    return df



# Common data type normalisation

# Formatting (trim whitespace, empty places become NULL)
def normalize_string_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            c = field.name
            df = df.withColumn(c, when(trim(col(c)) == "", None).otherwise(trim(col(c))))
    return df

# Replace boolean-like columns with actual boolean (Y / N become true and falls, etc...)
def normalize_boolean_columns(df, bool_cols):
    truthy = {"y", "yes", "true", "1"}
    falsy  = {"n", "no",  "false", "0"}

    for c in bool_cols:
        if c not in df.columns:
            continue
        lowered = lower(trim(col(c).cast("string")))
        df = df.withColumn(
            c,
            when(lowered.isin(*truthy), lit(True))
            .when(lowered.isin(*falsy),  lit(False))
            .otherwise(None)
            .cast(BooleanType())
        )
    return df

#Removes duplicates according to PK (or full row duplicates)
def perform_data_quality_checks(df, primary_keys):
    initial_count = df.count()

    if primary_keys:
        # 1. Drop row if ANY part of the composite primary key is missing (Null)
        df = df.dropna(subset=primary_keys)

        # 2. Drop duplicates based strictly on the combination of the primary keys
        df = df.dropDuplicates(subset=primary_keys)
    else:
        # If no primary key exists (like Taxi Trips), just drop exact full-row duplicates
        df = df.dropDuplicates()

    final_count = df.count()
    rejected_count = initial_count - final_count

    return df, initial_count, rejected_count


# For versioning in metadata
def schema_hash(df):
    return hash(tuple(sorted(df.dtypes)))


#-----------------------------Main Function-------------------
#steps follow what the assignment mentioned. 

def ingest_dataset(dataset_name, config):
    start_time = time.time()

    # Load according to file type (csv, parquet)
    df = spark.read.format(config["format"]).options(**config["options"]).load(config["path"])

    # Standardise column names (snake_case)
    df = standardize_columns(df)

    # Normalise common data types (remove whitespace)
    df = normalize_string_columns(df)

    #  Normalise timestamps to yyyy-mm-dd HH:mm:ss
    
    #  Columns already containing datetime strings in a known format
    df = normalize_timestamps(df, config.get("timestamp_cols", {}))
    #  Timestamp assembled from separate year/month/day/hour columns (weather)
    df = assemble_timestamp_from_parts(df, config.get("assembled_timestamp"))
    #  Timestamp assembled from separate date + time columns (air quality)
    df = normalize_date_time_pairs(df, config.get("date_time_pairs", []))

    # Normalise boolean data types
    df = normalize_boolean_columns(df, config.get("bool_cols", []))

    # quality checks (removing duplicates)
    standard_pks = [to_snake_case(c) for c in config["primary_keys"]]
    df, initial_cnt, rejected_cnt = perform_data_quality_checks(df, standard_pks)

    # Write to Delta
    output_path = f"./delta/{dataset_name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(output_path)


    execution_time = time.time() - start_time

    # Metadata
    metadata = {
        "dataset":                dataset_name,
        "processed_records":      initial_cnt,
        "rejected_records":       rejected_cnt,
        "final_records":          initial_cnt - rejected_cnt,
        "execution_time_seconds": round(execution_time, 2),
        "schema_version":         schema_hash(df),
    }
    return metadata

#-----------------------------RUN framework-------------------
ingestion_logs = []
for name, conf in DATASET_CONFIGS.items():
    print(f"Ingesting {name}...")
    log = ingest_dataset(name, conf)
    ingestion_logs.append(log)

print(*ingestion_logs, sep="\n")


Ingesting weather...


Ingesting taxi_trips_01...


Ingesting taxi_trips_02...


Ingesting taxi_trips_03...


Ingesting taxi_zone_lookup...
Ingesting air_quality...


{'dataset': 'weather', 'processed_records': 8784, 'rejected_records': 0, 'final_records': 8784, 'execution_time_seconds': 4.11, 'schema_version': 3056745397909432056}
{'dataset': 'taxi_trips_01', 'processed_records': 2964624, 'rejected_records': 0, 'final_records': 2964624, 'execution_time_seconds': 11.2, 'schema_version': 3320764916314618905}
{'dataset': 'taxi_trips_02', 'processed_records': 3007526, 'rejected_records': 1, 'final_records': 3007525, 'execution_time_seconds': 7.46, 'schema_version': 3320764916314618905}
{'dataset': 'taxi_trips_03', 'processed_records': 3582628, 'rejected_records': 0, 'final_records': 3582628, 'execution_time_seconds': 8.47, 'schema_version': 3320764916314618905}
{'dataset': 'taxi_zone_lookup', 'processed_records': 265, 'rejected_records': 0, 'final_records': 265, 'execution_time_seconds': -1.32, 'schema_version': 3907795482571684614}
{'dataset': 'air_quality', 'processed_records': 8139551, 'rejected_records': 0, 'final_records': 8139551, 'execution_time

In [9]:
df_delta1 = spark.read.format("delta").load("delta/taxi_trips_01")
df_delta2 = spark.read.format("delta").load("delta/weather")
df_delta3 = spark.read.format("delta").load("delta/taxi_zone_lookup")
df_delta4 = spark.read.format("delta").load("delta/air_quality")

df_delta1.show(1)
df_delta2.show(1)
df_delta3.show(1)
df_delta4.show(1)

+---------+--------------------+---------------------+---------------+-------------+-----------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|vendor_id|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|ratecode_id|store_and_fwd_flag|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+---------+--------------------+---------------------+---------------+-------------+-----------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|        2| 2024-01-01 01:57:55|  2024-01-01 02:17:43|              1|         1.72|          1|             false|           186|            79|           2|       17.7